# 3D FEM Simulation of the Krogh Cylinder

Every other notebook in this repo treats the Krogh cylinder as a 1D radial problem (the physically valid reduction, since the real model is axisymmetric — no angular or axial dependence). This notebook instead builds and solves the **real 3D geometry directly**: a solid cylindrical volume, meshed with tetrahedra, solved with the **Finite Element Method** (via [scikit-fem](https://scikit-fem.readthedocs.io/)) instead of the finite-difference/analytical solvers used elsewhere.

This is a qualitative, for-visualization demo, not a validated ground-truth solver — it hasn't been cross-checked against `analytical_transient_profile` the way `fdm_crank_nicolson` has in `data/synthetic/solver.py`. If you need trustworthy numbers, use the 1D solvers; use this one to *see* the diffusion happening in 3D.


## 1. Setup

In [ ]:
!pip install -q scikit-fem scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from scipy.spatial import Delaunay
from scipy.interpolate import griddata

import skfem
from skfem import Basis, ElementTetP1, asm, condense, solve
from skfem.models.poisson import laplace, mass

## 2. Physical parameters and mesh generation

Same PDE as everywhere else in the repo (`dc/dt = D*lap(c) - r*c`), same physical parameters. The mesh is a solid cylinder (radius `L`, height `H`) built by generating points on a structured cylindrical grid, then tetrahedralizing them with `scipy.spatial.Delaunay`. A small random jitter is added to the points — without it, the perfectly symmetric grid produces degenerate (zero-volume) tetrahedra, since many points are exactly cospherical and Delaunay's choice of how to split them becomes numerically ambiguous.


In [ ]:
D_phys, r_decay, c0, L = 10.0, 1e-3, 1.0, 100.0   # um^2/s, 1/s, concentration unit, um
H = 60.0                                          # cylinder height (um) - finite, for a tractable mesh

# WARNING: do not casually increase these to get a denser-looking result.
# The random jitter below (needed to avoid degenerate tets) makes some
# tetrahedra thin slivers, which occasionally makes the linear solve below
# catastrophically ill-conditioned. Bumping nr from 14 to 20 was measured to
# blow up the per-frame solve from ~17s total to ~15+ HOURS. Keep this mesh
# coarse and stable; get a denser/smoother look via interpolation onto a
# separate fine query cloud instead (section 4.5 below) - that's cheap and safe.
nr, ntheta, nz = 14, 24, 8                        # mesh resolution (keep as-is, see warning)

rng = np.random.default_rng(0)
r_vals = np.linspace(L / nr, L, nr)
theta_vals = np.linspace(0, 2 * np.pi, ntheta, endpoint=False)
z_vals = np.linspace(0, H, nz)

pts = []
for z in z_vals:
    pts.append((0.0, 0.0, z))  # central axis point
    for r in r_vals:
        for th in theta_vals:
            pts.append((r * np.cos(th), r * np.sin(th), z))
pts = np.array(pts)

jitter = rng.uniform(-1, 1, size=pts.shape) * (0.15 * L / nr)
pts_j = pts + jitter
pts_j[:, 2] = np.clip(pts_j[:, 2], 0, H)  # keep the top/bottom caps flat

tri = Delaunay(pts_j)
mesh = skfem.MeshTet(pts_j.T, tri.simplices.T)
print(f"mesh: {mesh.p.shape[1]} nodes, {mesh.t.shape[1]} tetrahedra")

## 3. Assemble FEM matrices and set the vessel boundary condition

`mass` and `laplace` are scikit-fem's built-in bilinear forms — no need to hand-derive the weak form. The vessel is modeled as a solid core: every mesh node within `r_cap` of the central axis is pinned at `c0` (Dirichlet). Every other boundary (the outer curved surface at `r=L`, and the top/bottom caps) is left alone, which in FEM automatically means zero-flux (Neumann) — the same "natural boundary condition" idea discussed for the 1D solver.


In [ ]:
basis = Basis(mesh, ElementTetP1())
M = asm(mass, basis)    # mass matrix
K = asm(laplace, basis)  # stiffness matrix

r_node = np.sqrt(mesh.p[0] ** 2 + mesh.p[1] ** 2)
r_cap = r_vals[0] * 1.5
inner_dofs = np.where(r_node <= r_cap)[0]
print(f"{len(inner_dofs)} / {mesh.p.shape[1]} nodes pinned as the vessel core (r <= {r_cap:.1f} um)")


## 4. Time-stepping (backward Euler)

`(M + dt*(D*K + r*M)) c_next = M c_cur`, unconditionally stable (same idea as the Crank-Nicolson scheme in `fdm_crank_nicolson`, but first-order in time instead of second). The vessel's Dirichlet condition is re-applied at every step via `skfem.condense`. Snapshots are saved at a handful of log-spaced times for the animation below.


In [ ]:
dt = 150.0
A = M + dt * (D_phys * K + r_decay * M)

c = np.zeros(mesh.p.shape[1])
c[inner_dofs] = c0
x_bc = c.copy()

n_frames = 40  # more frames = smoother animation over time; cheap since the mesh stays coarse
t_anim = np.logspace(1, np.log10(60000.0), n_frames)
frames = []
cur_t = 0.0
idx = 0
while idx < n_frames:
    while cur_t < t_anim[idx]:
        b = M @ c
        A_cond, b_cond, x_full, I = condense(A, b, x=x_bc, D=inner_dofs)
        c = solve(A_cond, b_cond, x_full, I)
        cur_t += dt
    frames.append(c.copy())
    idx += 1

print(f"Collected {len(frames)} snapshots, t from {t_anim[0]:,.0f}s to {t_anim[-1]:,.0f}s")

## 4.5 Denser point cloud, for a smoother-looking plot

The FEM mesh itself has to stay coarse (see the warning above). To get a visually denser, smoother-looking
result without touching the solver, this builds a much finer point cloud purely for plotting, and
**interpolates** the coarse FEM solution onto it with `scipy.interpolate.griddata` (linear interpolation over
the coarse mesh's own Delaunay triangulation — safe, since it's just interpolation weights, not a solve).
This is cheap (well under a second per frame) since it never rebuilds or re-solves anything.

In [ ]:
coarse_pts = mesh.p.T  # (n_nodes, 3)

# dense query cloud, for visualization only - no meshing/solving happens on this
nr_d, ntheta_d, nz_d = 40, 60, 20
r_d = np.linspace(0.5, L * 0.98, nr_d)
theta_d = np.linspace(0, 2 * np.pi, ntheta_d, endpoint=False)
z_d = np.linspace(H * 0.02, H * 0.98, nz_d)

qpts = np.array([
    (r * np.cos(th), r * np.sin(th), z)
    for z in z_d
    for r in r_d
    for th in theta_d
])
print(f"dense query cloud: {qpts.shape[0]} points (vs. {coarse_pts.shape[0]} FEM mesh nodes)")

dense_frames = [
    griddata(coarse_pts, frame, qpts, method="linear", fill_value=0.0)
    for frame in frames
]

## 5. Static snapshot — 3D cutaway view

Plotting every one of the ~3000 mesh nodes would hide the interior structure, so this keeps only the half with `y <= 0` — a cutaway that reveals the bright vessel core and the concentration fading outward toward the cylinder's surface.


In [ ]:
mask = qpts[:, 1] <= 1.5
x_m, y_m, z_m = qpts[mask, 0], qpts[mask, 1], qpts[mask, 2]

frame_idx = 20
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
sca = ax.scatter(x_m, y_m, z_m, c=dense_frames[frame_idx][mask], cmap="viridis", s=4, vmin=0, vmax=c0)
ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)"); ax.set_zlabel("z (um)")
ax.set_title(f"3D FEM cutaway, t={t_anim[frame_idx]:,.0f}s")
fig.colorbar(sca, shrink=0.6, label="C")

# save the figure to a PNG file right after creating it, while `fig` still
# refers to this exact figure (the animation cell below reuses the same
# `fig` variable name for a different figure, so save before moving on)
fig.savefig("fem_snapshot.png", dpi=150, bbox_inches="tight")
print("saved fem_snapshot.png")

plt.show()

## 6. Animation — the drug spreading through the 3D cylinder

Same cutaway view, animated through every collected snapshot.


In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
sca = ax.scatter(x_m, y_m, z_m, c=dense_frames[0][mask], cmap="viridis", s=4, vmin=0, vmax=c0)
ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)"); ax.set_zlabel("z (um)")
title = ax.set_title("")
fig.colorbar(sca, shrink=0.6, label="C")


def update(i):
    sca.set_array(dense_frames[i][mask])
    title.set_text(f"t={t_anim[i]:,.0f}s")
    return sca, title


anim = animation.FuncAnimation(fig, update, frames=len(dense_frames), interval=150, blit=False)

# writer="pillow" needs no extra system dependency (ffmpeg) - it re-runs
# `update()` for every frame and stitches the results into a .gif. This can
# take a little while (n_frames forward passes through matplotlib's renderer).
anim.save("fem_animation.gif", writer="pillow", fps=8)
print("saved fem_animation.gif")

plt.close(fig)
HTML(anim.to_jshtml())

## 7. Download the results

Both files (`fem_snapshot.png`, `fem_animation.gif`) were already saved to disk by the two cells above —
`savefig`/`anim.save` write to the current working directory, same as any normal Python file write. What
"download" means depends on where this notebook is running:

- **Google Colab**: the notebook's filesystem is a temporary remote VM, not your computer — `files.download(...)`
  (below) pushes the file through the browser as an actual download, the same as downloading anything else
  from a website.
- **Local Jupyter** (running on your own machine): the files are already sitting right there on your disk —
  "downloading" is meaningless, you'd just open the folder. The `except ImportError` branch below handles
  this by printing the path instead.

In [ ]:
import os

for path in ["fem_snapshot.png", "fem_animation.gif"]:
    print(f"{path}: {os.path.getsize(path) / 1024:.1f} KB")

try:
    from google.colab import files  # only importable inside a Colab runtime

    files.download("fem_snapshot.png")
    files.download("fem_animation.gif")
except ImportError:
    print("Not running in Colab - files are already saved locally at the paths above.")